# WEEK02 Scratch Models - smoke Linear / NLinear / DLinear

**Audience:** TA progress check (W2 2.1).

**Prerequisites:** `torch` installed; run from `Working_Files/` (or with that folder on `sys.path`).

**Learning goals:** Confirm all three models forward on a tiny synthetic tensor; shapes `(B, C, L) → (B, C, H)`; weight counts ≈ `L·H` (Linear/NLinear) and `2·L·H` (DLinear), ignoring bias.

**Note:** Production code lives in `src/models.py`. This notebook only smoke-tests imports - it does not reimplement the models.

## Outline

1. Setup + import from `src`
2. Synthetic tensor forward (all three)
3. Param-count checks (shared W)
4. Optional: NLinear recenter sanity

## 1. Setup

Layout is **channel-first** `(B, C, L)` so `MovingAvg` can use `AvgPool1d`. Paper tables often write `(B, L, C)` — same tensor after a permute.

In [2]:
from pathlib import Path
import sys

import torch

from src.models import DLinear, Linear, NLinear

ROOT = Path.cwd()
if not (ROOT / "src" / "models.py").exists():
    # if kernel cwd is notebooks/ or elsewhere, walk up once
    ROOT = ROOT.parent if (ROOT.parent / "src" / "models.py").exists() else ROOT
sys.path.insert(0, str(ROOT))


torch.manual_seed(21)
B, C, L, H = 4, 3, 16, 8
x = torch.randn(B, C, L)
ROOT, x.shape

(PosixPath('/mnt/c/HOCTAP/AIO_2026/module4/research/DLinear_NLinear/Working_Files'),
 torch.Size([4, 3, 16]))

## 2. Forward smoke — all three

Expect `(B, C, H) = (4, 3, 8)` for each model (shared weights).

In [3]:
models = {
    "Linear": Linear(seq_len=L, pred_len=H, individual=False),
    "NLinear": NLinear(seq_len=L, pred_len=H, individual=False),
    "DLinear": DLinear(seq_len=L, pred_len=H, kernel=3, individual=False),
}

for name, model in models.items():
    y = model(x)
    assert y.shape == (B, C, H), (name, y.shape)
    print(f"{name:8s}  {tuple(x.shape)} → {tuple(y.shape)}")

print("OK: all three forward shapes")

Linear    (4, 3, 16) → (4, 3, 8)
NLinear   (4, 3, 16) → (4, 3, 8)
DLinear   (4, 3, 16) → (4, 3, 8)
OK: all three forward shapes


## 3. Param counts (shared W, ignore bias)

| Model | Expected weight count |
| ----- | --------------------- |
| Linear / NLinear | `L · H` |
| DLinear | `2 · L · H` (seasonal + trend heads) |

In [4]:
def n_weight(m: torch.nn.Module) -> int:
    return sum(p.numel() for n, p in m.named_parameters() if "bias" not in n)


assert n_weight(models["Linear"]) == L * H
assert n_weight(models["NLinear"]) == L * H
assert n_weight(models["DLinear"]) == 2 * L * H

for name, model in models.items():
    print(
        f"{name:8s}  weights={n_weight(model):5d}  (expect L·H={L * H} or 2·L·H={2 * L * H})"
    )

print("OK: param counts")

Linear    weights=  128  (expect L·H=128 or 2·L·H=256)
NLinear   weights=  128  (expect L·H=128 or 2·L·H=256)
DLinear   weights=  256  (expect L·H=128 or 2·L·H=256)
OK: param counts


## 4. NLinear recenter

With zero weights/bias, forecast equals the last lookback value (add-back `l`).

In [5]:
nl = NLinear(seq_len=L, pred_len=H, individual=False)
with torch.no_grad():
    nl.Linear.weight.zero_()
    nl.Linear.bias.zero_()

y = nl(x)
expected = x[:, :, -1:].expand(B, C, H)
# check if y and expected are the same
assert torch.allclose(y, expected)
print("OK: NLinear add-back (zero W → last value)")

OK: NLinear add-back (zero W → last value)


## Done

If every cell printed `OK`, §2.1 smoke is satisfied for TA review. Automated coverage also lives in `test_src/test_models.py` (`pytest`).